# 05 — Attention and Context: How Transformers See and Remember> **Orbit -1 (Foundations)** · **Domain**: Foundations · **Difficulty**: Intermediate · **Reading time**: ~35 min[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/castalia/blob/main/foundations/05_attention_and_context.ipynb)**Prerequisites**:- [00_how_llms_work.ipynb](00_how_llms_work.ipynb) — The LLM pipeline- [02_embeddings_and_vector_spaces.ipynb](02_embeddings_and_vector_spaces.ipynb) — Vector similarity**What you will learn**:- Self-attention: how each token "looks at" every other token, implemented from scratch- Multi-head attention: why multiple perspectives matter- The KV cache: why it exists and how it speeds up generation- Context windows: what limits them and what happens at the boundary- Positional encodings: how transformers know word order- Why this matters for prompt engineering, RAG context budgets, and serving systems

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **The problem attention solvesBefore transformers, recurrent neural networks (RNNs) processed text **one token at a time**, left to right.Each step folded the current token into a fixed-size hidden state — a single vector that had to carry*everything* the network remembered about the prior context.This is like reading a novel while compressing every chapter into a single sticky note. By chapter 20, thenote is hopelessly blurred. Long-range dependencies — a pronoun referring to a character introduced 500words ago — are lost.### The bottleneck```Token 1 ──▶ h₁ ──▶ Token 2 ──▶ h₂ ──▶ … ──▶ Token N ──▶ hN              │                   │                         │        (all context       (all context               (all context         compressed)        compressed)                compressed)```**Attention breaks this bottleneck.** Instead of forcing information through a single hidden state,attention lets every token *directly* look at every other token and decide how much to attend to each.No compression, no information loss — just weighted retrieval.**
- Understand **Self-attention from scratchReal attention differs from our toy lookup in one critical way: instead of using the raw embeddings asquery, key, and value, the transformer **projects** each embedding into three separate spaces usinglearned weight matrices.For an input matrix $X \in \mathbb{R}^{n \times d}$ (n tokens, d dimensions):$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$Then the attention output is:$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$Where $d_k$ is the dimension of the key vectors. Let's implement every step.**
- Understand **Multi-head attention: multiple perspectivesA single attention head captures one type of relationship. But language has many overlapping structures:syntactic (subject→verb), semantic (synonym clusters), positional (nearby words), coreference (pronoun→noun).**Multi-head attention** runs several attention heads in parallel, each with its own $W_Q$, $W_K$, $W_V$projections. Each head operates on a *slice* of the embedding dimension, so the total computation isroughly the same as single-head attention on the full dimension.$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$where $\text{head}_i = \text{Attention}(X W_Q^i, X W_K^i, X W_V^i)$ and each projection maps from$d_{\text{model}}$ to $d_k = d_{\text{model}} / h$.**
- Understand **Causal masking: why LLMs can only look backwardDuring text generation, token $n$ is produced *before* tokens $n+1, n+2, \ldots$ even exist.So at training time, we must prevent the model from "cheating" by looking at future tokens.The solution: a **causal mask** — a lower-triangular matrix that sets all future positions to $-\infty$before the softmax. Since $\text{softmax}(-\infty) = 0$, those positions get zero attention weight.```              t₁   t₂   t₃   t₄   t₅  t₁  →    [  0   -∞   -∞   -∞   -∞  ]     ← can only see itself  t₂  →    [  0    0   -∞   -∞   -∞  ]     ← can see t₁, t₂  t₃  →    [  0    0    0   -∞   -∞  ]     ← can see t₁, t₂, t₃  t₄  →    [  0    0    0    0   -∞  ]  t₅  →    [  0    0    0    0    0  ]     ← can see everything```This is why LLMs generate left-to-right: each token's representation depends only on what came before.**
- Understand **The KV cache: why inference gets fasterDuring autoregressive generation, the model produces one token at a time. At step $t$, it runs attentionover the full sequence of $t$ tokens. Naively, this means recomputing $K$ and $V$ for all $t$ tokens atevery step — even though the previous $t-1$ tokens haven't changed.The **KV cache** stores the $K$ and $V$ projections from all prior tokens. At step $t$:1. Compute $Q_t$, $K_t$, $V_t$ for the **new token only**.2. Append $K_t$, $V_t$ to the cache.3. Run attention: $Q_t$ against the **full cached K and V**.This reduces per-step computation from $O(t \cdot d)$ to $O(d)$ for the projections, and the attentionitself is just one row of the attention matrix instead of the full $t \times t$ matrix.### The memory trade-offKV cache memory for one sequence:$$\text{KV memory} = 2 \times L \times n_h \times d_h \times s \times b$$where $L$ = layers, $n_h$ = heads, $d_h$ = head dimension, $s$ = sequence length, $b$ = bytes per element.This is a **linear** cost in sequence length — but the constants are large. For a 8B-parameter model at128K context, the KV cache alone can exceed the model weights in memory.**


### 📊 Visual: Self-Attention Mechanism

In [ ]:
# Visual: Self-Attention Weight Matrix
import matplotlib.pyplot as plt
import numpy as np

tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']
n = len(tokens)
np.random.seed(42)

# Simulate attention weights (rows attend to columns)
raw = np.random.randn(n, n)
# Make it more realistic: tokens attend to nearby + semantically related
for i in range(n):
    for j in range(n):
        raw[i][j] -= abs(i - j) * 0.5  # distance penalty
raw[1][5] += 2  # "cat" attends to "mat"
raw[3][1] += 1.5  # "on" attends to "cat"

# Softmax
exp = np.exp(raw - raw.max(axis=1, keepdims=True))
attn = exp / exp.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn, cmap='YlOrRd', vmin=0, vmax=0.5)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Key (attending to)', fontsize=11)
ax.set_ylabel('Query (from)', fontsize=11)
ax.set_title('Self-Attention Weights', fontsize=13, fontweight='bold')

for i in range(n):
    for j in range(n):
        color = 'white' if attn[i][j] > 0.3 else 'black'
        ax.text(j, i, f'{attn[i][j]:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

In [ ]:
# @title Setup — Run this cell first# --- Foundations Notebook Setup ---!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch numpy matplotlibimport mathimport timeimport warningsfrom pathlib import Pathimport matplotlib.pyplot as pltimport matplotlib.ticker as tickerimport numpy as npimport torchimport torch.nn.functional as Fwarnings.filterwarnings("ignore", category=FutureWarning)plt.style.use("seaborn-v0_8-whitegrid")np.random.seed(42)torch.manual_seed(42)try:    from google.colab import drive    drive.mount("/content/drive")    CACHE_DIR = Path("/content/drive/MyDrive/models")except Exception:    CACHE_DIR = Path("./models")CACHE_DIR.mkdir(parents=True, exist_ok=True)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"✓ Foundations notebook environment ready")print(f"  Device : {DEVICE}")print(f"  Cache  : {CACHE_DIR}")if DEVICE == "cuda":    gpu_name = torch.cuda.get_device_name(0)    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9    print(f"  GPU    : {gpu_name} ({vram_gb:.1f} GB)")

## 1 — The problem attention solvesBefore transformers, recurrent neural networks (RNNs) processed text **one token at a time**, left to right.Each step folded the current token into a fixed-size hidden state — a single vector that had to carry*everything* the network remembered about the prior context.This is like reading a novel while compressing every chapter into a single sticky note. By chapter 20, thenote is hopelessly blurred. Long-range dependencies — a pronoun referring to a character introduced 500words ago — are lost.### The bottleneck```Token 1 ──▶ h₁ ──▶ Token 2 ──▶ h₂ ──▶ … ──▶ Token N ──▶ hN              │                   │                         │        (all context       (all context               (all context         compressed)        compressed)                compressed)```**Attention breaks this bottleneck.** Instead of forcing information through a single hidden state,attention lets every token *directly* look at every other token and decide how much to attend to each.No compression, no information loss — just weighted retrieval.

### Attention as lookupThe simplest mental model: attention is a **soft dictionary lookup**.- You have a **query** (the word you are currently processing).- You have a set of **keys** (all the words in the context).- Each key has a **value** (the information that word contributes).The query checks similarity against every key. High-similarity keys contribute more of their value tothe output. The result is a *weighted average* of all values, where the weights are the attention scores.Let's see this concretely.

In [ ]:
# A toy attention lookup: which context words does our query attend to?# Suppose each word is represented by a simple 4-d vector (in reality, these# are learned embeddings of dimension 768, 4096, etc.)words   = ["The", "cat", "sat", "on", "the", "mat"]# Pretend embeddings — just random vectors for illustrationnp.random.seed(7)embeddings = {w: np.random.randn(4) for w in words}query_word = "sat"query_vec  = embeddings[query_word]# Compute raw dot-product similarity against every context wordscores = [np.dot(query_vec, embeddings[w]) for w in words]# Softmax — turn raw scores into a probability distributiondef softmax(x):    """Numerically stable softmax."""    e = np.exp(x - np.max(x))    return e / e.sum()weights = softmax(np.array(scores))print(f"Query: \"{query_word}\"\n")print(f"  {"Token":<6}  {"Raw score":>10}  {"Attn weight":>11}")print(f"  {"─"*6}  {"─"*10}  {"─"*11}")for w, s, a in zip(words, scores, weights):    bar = "█" * int(a * 40)    print(f"  {w:<6}  {s:>10.3f}  {a:>11.3f}  {bar}")print("\n→ This IS attention: a softmax-weighted lookup over all context words.")

### Attention pattern for a full sentenceIn self-attention, **every** token acts as both a query and a key. The result is an *attention matrix*where entry (i, j) tells us how much token i attends to token j.```             The   cat   sat   on   the   mat  The      [ 0.3   0.2   0.1  0.1  0.2   0.1 ]  cat      [ 0.1   0.4   0.2  0.1  0.1   0.1 ]  sat      [ 0.1   0.3   0.2  0.2  0.1   0.1 ]  on       [ 0.1   0.1   0.2  0.3  0.1   0.2 ]  the      [ 0.2   0.1   0.1  0.1  0.3   0.2 ]  mat      [ 0.1   0.1   0.1  0.2  0.2   0.3 ]```Each row sums to 1 (softmax). Each row is one token's "view" of the whole sequence.We will implement this from scratch in the next section.

## 2 — Self-attention from scratchReal attention differs from our toy lookup in one critical way: instead of using the raw embeddings asquery, key, and value, the transformer **projects** each embedding into three separate spaces usinglearned weight matrices.For an input matrix $X \in \mathbb{R}^{n \times d}$ (n tokens, d dimensions):$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$Then the attention output is:$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$Where $d_k$ is the dimension of the key vectors. Let's implement every step.

In [ ]:
# ── Self-attention from scratch (NumPy) ──────────────────────────────────np.random.seed(42)# Parametersn_tokens = 6               # sequence lengthd_model  = 8               # embedding dimensiond_k      = d_model          # key/query dimension (often = d_model)d_v      = d_model          # value dimension# Step 0: Input embeddings (normally these come from the embedding table)tokens = ["The", "cat", "sat", "on", "the", "mat"]X = np.random.randn(n_tokens, d_model)  # (6, 8)print(f"Input X shape: {X.shape}  (n_tokens × d_model)")# Step 1: Create projection matricesW_Q = np.random.randn(d_model, d_k) * 0.1W_K = np.random.randn(d_model, d_k) * 0.1W_V = np.random.randn(d_model, d_v) * 0.1# Step 2: Project to Q, K, VQ = X @ W_Q   # (6, 8)K = X @ W_KV = X @ W_Vprint(f"Q shape: {Q.shape}   K shape: {K.shape}   V shape: {V.shape}")# Step 3: Compute raw attention scoresscores = Q @ K.T          # (6, 6)print(f"\nRaw scores shape: {scores.shape}")print(f"Raw scores (before scaling):\n{np.round(scores, 2)}")# Step 4: Scale by sqrt(d_k)scale = np.sqrt(d_k)scaled_scores = scores / scaleprint(f"\nScaling factor: sqrt({d_k}) = {scale:.2f}")# Step 5: Apply softmax row-wisedef softmax_matrix(M):    """Row-wise softmax for a 2D matrix."""    e = np.exp(M - M.max(axis=-1, keepdims=True))    return e / e.sum(axis=-1, keepdims=True)attn_weights = softmax_matrix(scaled_scores)  # (6, 6)print(f"\nAttention weights (each row sums to 1):")print(np.round(attn_weights, 3))print(f"Row sums: {np.round(attn_weights.sum(axis=1), 3)}")# Step 6: Weighted sum of valuesoutput = attn_weights @ V   # (6, 8)print(f"\nOutput shape: {output.shape}  (same as input — each token gets a new representation)")

In [ ]:
# ── Visualize the attention weight matrix ────────────────────────────────fig, ax = plt.subplots(figsize=(6, 5))im = ax.imshow(attn_weights, cmap="Blues", vmin=0, vmax=attn_weights.max())ax.set_xticks(range(n_tokens))ax.set_yticks(range(n_tokens))ax.set_xticklabels(tokens, fontsize=11)ax.set_yticklabels(tokens, fontsize=11)ax.set_xlabel("Key (attended to)", fontsize=12)ax.set_ylabel("Query (attending)", fontsize=12)ax.set_title("Self-Attention Weights", fontsize=14, fontweight="bold")# Annotate cellsfor i in range(n_tokens):    for j in range(n_tokens):        val = attn_weights[i, j]        color = "white" if val > 0.25 else "black"        ax.text(j, i, f"{val:.2f}", ha="center", va="center",                fontsize=9, color=color)fig.colorbar(im, ax=ax, shrink=0.8)plt.tight_layout()plt.show()print("Each row is one token's attention distribution over all tokens.")print("Brighter cells = stronger attention.")

### Why the $\sqrt{d_k}$ scaling mattersWithout scaling, as $d_k$ grows the dot products grow in magnitude. Large dot products push softmax intoregions where it is nearly one-hot (saturated) — gradients vanish and the model stops learning.The scaling factor keeps the variance of the dot products at ~1 regardless of dimension, ensuring softmaxstays in a useful gradient regime. This is a small detail with a big impact.

In [ ]:
# ── Demonstrate softmax saturation without scaling ───────────────────────dims = [8, 64, 512, 4096]fig, axes = plt.subplots(1, len(dims), figsize=(16, 3.5), sharey=True)for ax, d in zip(axes, dims):    np.random.seed(0)    q = np.random.randn(d)    K_rand = np.random.randn(10, d)    raw   = K_rand @ q           # unscaled    scaled = raw / np.sqrt(d)     # scaled    softmax_raw    = softmax(raw)    softmax_scaled = softmax(scaled)    x = np.arange(10)    ax.bar(x - 0.15, softmax_raw,    width=0.3, label="Unscaled", alpha=0.8)    ax.bar(x + 0.15, softmax_scaled, width=0.3, label="Scaled",   alpha=0.8)    ax.set_title(f"d_k = {d}", fontsize=11)    ax.set_xlabel("Key index")    if d == 8:        ax.set_ylabel("Softmax weight")    ax.legend(fontsize=8)fig.suptitle("Softmax saturation: unscaled vs scaled attention scores",             fontsize=13, fontweight="bold", y=1.03)plt.tight_layout()plt.show()print("At d_k = 4096, unscaled softmax is nearly one-hot — only one key gets attention.")print("Scaling keeps the distribution smooth, allowing the model to learn nuanced patterns.")

## 3 — Multi-head attention: multiple perspectivesA single attention head captures one type of relationship. But language has many overlapping structures:syntactic (subject→verb), semantic (synonym clusters), positional (nearby words), coreference (pronoun→noun).**Multi-head attention** runs several attention heads in parallel, each with its own $W_Q$, $W_K$, $W_V$projections. Each head operates on a *slice* of the embedding dimension, so the total computation isroughly the same as single-head attention on the full dimension.$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$where $\text{head}_i = \text{Attention}(X W_Q^i, X W_K^i, X W_V^i)$ and each projection maps from$d_{\text{model}}$ to $d_k = d_{\text{model}} / h$.

In [ ]:
# ── Multi-head attention from scratch ────────────────────────────────────np.random.seed(42)n_tokens = 6d_model  = 16            # larger so we can split across headsn_heads  = 4d_k_head = d_model // n_heads   # 4 dims per headtokens = ["The", "cat", "sat", "on", "the", "mat"]X = np.random.randn(n_tokens, d_model)def single_head_attention(X, W_Q, W_K, W_V):    """Compute scaled dot-product attention for one head."""    Q = X @ W_Q    K = X @ W_K    V = X @ W_V    scores = Q @ K.T / np.sqrt(W_Q.shape[1])    weights = softmax_matrix(scores)    output = weights @ V    return output, weights# Create weight matrices for each headall_outputs = []all_weights = []for h in range(n_heads):    W_Q_h = np.random.randn(d_model, d_k_head) * 0.1    W_K_h = np.random.randn(d_model, d_k_head) * 0.1    W_V_h = np.random.randn(d_model, d_k_head) * 0.1    out_h, weights_h = single_head_attention(X, W_Q_h, W_K_h, W_V_h)    all_outputs.append(out_h)    all_weights.append(weights_h)# Concatenate head outputsconcat = np.concatenate(all_outputs, axis=-1)  # (6, 16)# Final linear projectionW_O = np.random.randn(d_model, d_model) * 0.1multi_head_output = concat @ W_Oprint(f"Per-head output shape : {all_outputs[0].shape}  (n_tokens × d_k_head)")print(f"Concatenated shape    : {concat.shape}       (n_tokens × d_model)")print(f"Final output shape    : {multi_head_output.shape}       (same as input)")

In [ ]:
# ── Visualize each head's attention pattern ──────────────────────────────fig, axes = plt.subplots(1, n_heads, figsize=(18, 4))for h, ax in enumerate(axes):    im = ax.imshow(all_weights[h], cmap="Blues", vmin=0, vmax=0.5)    ax.set_xticks(range(n_tokens))    ax.set_yticks(range(n_tokens))    ax.set_xticklabels(tokens, fontsize=9, rotation=45)    ax.set_yticklabels(tokens, fontsize=9)    ax.set_title(f"Head {h}", fontsize=12, fontweight="bold")    for i in range(n_tokens):        for j in range(n_tokens):            val = all_weights[h][i, j]            color = "white" if val > 0.25 else "black"            ax.text(j, i, f"{val:.2f}", ha="center", va="center",                    fontsize=7, color=color)fig.suptitle("Multi-Head Attention — Each head learns a different pattern",             fontsize=14, fontweight="bold")plt.tight_layout()plt.show()print("Different heads attend to different positions — that is the point.")print("One head might focus on the previous word, another on the subject, etc.")

### Real attention from a pretrained modelLet's load a real model and extract its attention weights to see what learned heads actually look like.We use Qwen3-8B (4-bit quantized to fit on a T4 GPU's 16 GB VRAM).

In [ ]:
# ── Load Qwen3-8B (4-bit quantized) ─────────────────────────────────────from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfigMODEL_ID = "Qwen/Qwen3-8B"bnb_config = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_compute_dtype=torch.float16,    bnb_4bit_quant_type="nf4",)tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)model = AutoModelForCausalLM.from_pretrained(    MODEL_ID,    quantization_config=bnb_config,    device_map="auto",    cache_dir=CACHE_DIR,    torch_dtype=torch.float16,)model.eval()if DEVICE == "cuda":    vram_used = torch.cuda.memory_allocated() / 1e9    print(f"✓ Model loaded — VRAM used: {vram_used:.1f} GB")else:    print("✓ Model loaded (CPU mode — will be slow)")

In [ ]:
# ── Extract real attention weights ───────────────────────────────────────prompt = "The capital of France is Paris, which is known for"inputs = tokenizer(prompt, return_tensors="pt").to(model.device)tok_labels = [tokenizer.decode(t) for t in inputs["input_ids"][0]]with torch.no_grad():    outputs = model(**inputs, output_attentions=True)# outputs.attentions is a tuple: one tensor per layer# Each tensor shape: (batch, n_heads, seq_len, seq_len)n_layers = len(outputs.attentions)n_heads_model = outputs.attentions[0].shape[1]seq_len = outputs.attentions[0].shape[2]print(f"Model has {n_layers} layers × {n_heads_model} heads")print(f"Sequence length: {seq_len} tokens")print(f"Tokens: {tok_labels}")

In [ ]:
# ── Visualize 4 heads from different layers ──────────────────────────────# Pick heads from different layers to show diversitylayer_head_pairs = [    (0, 0, "Layer 0, Head 0 (early)"),    (n_layers // 4, 2, f"Layer {n_layers // 4}, Head 2"),    (n_layers // 2, 0, f"Layer {n_layers // 2}, Head 0 (middle)"),    (n_layers - 1, 0, f"Layer {n_layers - 1}, Head 0 (final)"),]fig, axes = plt.subplots(1, 4, figsize=(20, 5))for ax, (layer, head, title) in zip(axes, layer_head_pairs):    attn = outputs.attentions[layer][0, head].cpu().float().numpy()    im = ax.imshow(attn, cmap="magma", vmin=0)    ax.set_xticks(range(seq_len))    ax.set_yticks(range(seq_len))    ax.set_xticklabels(tok_labels, fontsize=7, rotation=90)    ax.set_yticklabels(tok_labels, fontsize=7)    ax.set_title(title, fontsize=10, fontweight="bold")fig.suptitle("Real Attention Heads from Qwen3-8B — Each head sees different patterns",             fontsize=13, fontweight="bold")plt.tight_layout()plt.show()print("Notice: early layers often have 'local' attention (nearby tokens).")print("Later layers show more 'semantic' attention (content-based, long-range).")print("The causal mask is visible — no attention to future tokens (upper triangle is dark).")

## 4 — Causal masking: why LLMs can only look backwardDuring text generation, token $n$ is produced *before* tokens $n+1, n+2, \ldots$ even exist.So at training time, we must prevent the model from "cheating" by looking at future tokens.The solution: a **causal mask** — a lower-triangular matrix that sets all future positions to $-\infty$before the softmax. Since $\text{softmax}(-\infty) = 0$, those positions get zero attention weight.```              t₁   t₂   t₃   t₄   t₅  t₁  →    [  0   -∞   -∞   -∞   -∞  ]     ← can only see itself  t₂  →    [  0    0   -∞   -∞   -∞  ]     ← can see t₁, t₂  t₃  →    [  0    0    0   -∞   -∞  ]     ← can see t₁, t₂, t₃  t₄  →    [  0    0    0    0   -∞  ]  t₅  →    [  0    0    0    0    0  ]     ← can see everything```This is why LLMs generate left-to-right: each token's representation depends only on what came before.

In [ ]:
# ── Implement causal masking ─────────────────────────────────────────────np.random.seed(42)n = 6d = 8tokens = ["The", "cat", "sat", "on", "the", "mat"]X = np.random.randn(n, d)W_Q = np.random.randn(d, d) * 0.1W_K = np.random.randn(d, d) * 0.1W_V = np.random.randn(d, d) * 0.1Q = X @ W_QK = X @ W_KV = X @ W_Vscores = Q @ K.T / np.sqrt(d)# Create causal mask: lower triangular = 0, upper triangular = -infcausal_mask = np.triu(np.ones((n, n)) * (-1e9), k=1)print("Causal mask (0 = allowed, -1e9 = blocked):")print(np.where(causal_mask == 0, "  0", " -∞"))# Apply mask before softmaxmasked_scores = scores + causal_mask# Unmasked vs masked attention weightsattn_unmasked = softmax_matrix(scores)attn_masked   = softmax_matrix(masked_scores)print(f"\nUnmasked — \"sat\" attends to:  {dict(zip(tokens, np.round(attn_unmasked[2], 3)))}")print(f"Masked   — \"sat\" attends to:  {dict(zip(tokens, np.round(attn_masked[2],   3)))}")print("\n→ With the mask, \"sat\" cannot see \"on\", \"the\", or \"mat\".")

In [ ]:
# ── Side-by-side: masked vs unmasked attention ───────────────────────────fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))for ax, weights, title in [    (ax1, attn_unmasked, "Unmasked (bidirectional)"),    (ax2, attn_masked,   "Causal-masked (autoregressive)"),]:    im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=0.5)    ax.set_xticks(range(n))    ax.set_yticks(range(n))    ax.set_xticklabels(tokens, fontsize=10)    ax.set_yticklabels(tokens, fontsize=10)    ax.set_title(title, fontsize=12, fontweight="bold")    ax.set_xlabel("Key")    ax.set_ylabel("Query")    for i in range(n):        for j in range(n):            val = weights[i, j]            color = "white" if val > 0.25 else "black"            ax.text(j, i, f"{val:.2f}", ha="center", va="center",                    fontsize=8, color=color)fig.suptitle("Causal Masking — LLMs can only attend to past and present tokens",             fontsize=14, fontweight="bold")plt.tight_layout()plt.show()print("Left: bidirectional (used in BERT-style models for understanding tasks).")print("Right: causal (used in GPT/Qwen/Llama for generation). Upper triangle is zeroed out.")

## 5 — The KV cache: why inference gets fasterDuring autoregressive generation, the model produces one token at a time. At step $t$, it runs attentionover the full sequence of $t$ tokens. Naively, this means recomputing $K$ and $V$ for all $t$ tokens atevery step — even though the previous $t-1$ tokens haven't changed.The **KV cache** stores the $K$ and $V$ projections from all prior tokens. At step $t$:1. Compute $Q_t$, $K_t$, $V_t$ for the **new token only**.2. Append $K_t$, $V_t$ to the cache.3. Run attention: $Q_t$ against the **full cached K and V**.This reduces per-step computation from $O(t \cdot d)$ to $O(d)$ for the projections, and the attentionitself is just one row of the attention matrix instead of the full $t \times t$ matrix.### The memory trade-offKV cache memory for one sequence:$$\text{KV memory} = 2 \times L \times n_h \times d_h \times s \times b$$where $L$ = layers, $n_h$ = heads, $d_h$ = head dimension, $s$ = sequence length, $b$ = bytes per element.This is a **linear** cost in sequence length — but the constants are large. For a 8B-parameter model at128K context, the KV cache alone can exceed the model weights in memory.

In [ ]:
# ── Simulating the KV cache ──────────────────────────────────────────────np.random.seed(42)d = 8seq_len = 10tokens_seq = [f"tok_{i}" for i in range(seq_len)]# Full sequence embeddingsX_full = np.random.randn(seq_len, d)# Weight matricesW_Q = np.random.randn(d, d) * 0.1W_K = np.random.randn(d, d) * 0.1W_V = np.random.randn(d, d) * 0.1# -- WITHOUT cache: recompute K, V for all tokens at each step --total_ops_no_cache = 0for t in range(1, seq_len + 1):    # Must project ALL t tokens to get K, V    Q = X_full[:t] @ W_Q  # (t, d)    K = X_full[:t] @ W_K  # (t, d)    V = X_full[:t] @ W_V  # (t, d)    total_ops_no_cache += 3 * t * d * d  # 3 projections, each t×d @ d×d# -- WITH cache: only compute K, V for the new token --K_cache = np.zeros((0, d))V_cache = np.zeros((0, d))total_ops_cache = 0for t in range(seq_len):    x_new = X_full[t:t+1]                # (1, d) — just the new token    q_new = x_new @ W_Q                   # (1, d)    k_new = x_new @ W_K                   # (1, d)    v_new = x_new @ W_V                   # (1, d)    K_cache = np.vstack([K_cache, k_new])  # grow cache    V_cache = np.vstack([V_cache, v_new])    # Attention: q_new (1, d) × K_cache.T (d, t+1) → (1, t+1)    total_ops_cache += 3 * 1 * d * d      # 3 projections for 1 tokenspeedup = total_ops_no_cache / total_ops_cacheprint(f"Projection ops WITHOUT cache: {total_ops_no_cache:,}")print(f"Projection ops WITH cache:    {total_ops_cache:,}")print(f"Speedup from caching: {speedup:.1f}× fewer projection operations")print(f"\nCache size at end: K={K_cache.shape}, V={V_cache.shape}")

In [ ]:
# ── KV cache memory calculator for real models ──────────────────────────def kv_cache_memory_gb(    n_layers: int,    n_heads: int,    head_dim: int,    seq_len: int,    dtype_bytes: int = 2,  # fp16 = 2 bytes    n_kv_heads: int = None,  # for GQA; None = same as n_heads) -> float:    """Compute KV cache memory in GB for one sequence."""    if n_kv_heads is None:        n_kv_heads = n_heads    # 2 for K and V, then layers × kv_heads × head_dim × seq_len × bytes    bytes_total = 2 * n_layers * n_kv_heads * head_dim * seq_len * dtype_bytes    return bytes_total / (1024 ** 3)# Model configsmodels = {    "Qwen3-8B":    dict(n_layers=36, n_heads=32, head_dim=128, n_kv_heads=8),    "Llama-3-8B":  dict(n_layers=32, n_heads=32, head_dim=128, n_kv_heads=8),    "Llama-3-70B": dict(n_layers=80, n_heads=64, head_dim=128, n_kv_heads=8),}seq_lengths = [1024, 4096, 8192, 32768, 65536, 131072]print(f"{"Model":<16} {"Seq Len":>8}  {"KV Cache (fp16)":>15}  {"KV Cache (fp8)":>14}")print("─" * 60)for name, cfg in models.items():    for s in seq_lengths:        mem_fp16 = kv_cache_memory_gb(**cfg, seq_len=s, dtype_bytes=2)        mem_fp8  = kv_cache_memory_gb(**cfg, seq_len=s, dtype_bytes=1)        flag = " ⚠️ > T4 VRAM!" if mem_fp16 > 15 else ""        print(f"{name:<16} {s:>8,}  {mem_fp16:>12.2f} GB  {mem_fp8:>11.2f} GB{flag}")    print()

In [ ]:
# ── Plot KV cache memory vs sequence length ──────────────────────────────fig, ax = plt.subplots(figsize=(10, 5))seq_range = np.arange(1024, 131073, 1024)colors = ["#2196F3", "#FF9800", "#E91E63"]for (name, cfg), color in zip(models.items(), colors):    mem = [kv_cache_memory_gb(**cfg, seq_len=int(s)) for s in seq_range]    ax.plot(seq_range / 1000, mem, label=name, linewidth=2, color=color)ax.axhline(y=16, color="red", linestyle="--", alpha=0.7, label="T4 VRAM (16 GB)")ax.set_xlabel("Sequence length (thousands of tokens)", fontsize=12)ax.set_ylabel("KV cache memory (GB)", fontsize=12)ax.set_title("KV Cache Memory vs Sequence Length (fp16, single sequence)",             fontsize=13, fontweight="bold")ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Key insight: for long contexts, KV cache memory can exceed model weights.")print("This is why systems like vLLM use paged attention — to manage KV cache like virtual memory.")

## 6 — Context windows: what limits them and whyA model's **context window** is the maximum number of tokens it can process in a single forward pass.Two factors make long contexts expensive:1. **Attention is $O(n^2)$** in sequence length — every token attends to every other token.   Doubling the context quadruples the attention computation.2. **KV cache is $O(n)$** — linear but with large constants (as we just saw).Together, these create practical limits even when the model *architecture* allows a longer context.

In [ ]:
# ── Quadratic scaling of attention ───────────────────────────────────────seq_lengths_plot = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])# Relative computation (normalized to seq_len=512)attention_compute = (seq_lengths_plot ** 2) / (512 ** 2)kv_memory         = seq_lengths_plot / 512fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# Attention computeax1.plot(seq_lengths_plot / 1000, attention_compute, "o-", color="#E91E63",         linewidth=2, markersize=5)ax1.set_xlabel("Sequence length (K tokens)", fontsize=11)ax1.set_ylabel("Relative compute (vs 512 tokens)", fontsize=11)ax1.set_title("Attention compute: O(n²)", fontsize=12, fontweight="bold")ax1.set_yscale("log")ax1.grid(True, alpha=0.3)# KV cache memoryax2.plot(seq_lengths_plot / 1000, kv_memory, "s-", color="#2196F3",         linewidth=2, markersize=5)ax2.set_xlabel("Sequence length (K tokens)", fontsize=11)ax2.set_ylabel("Relative memory (vs 512 tokens)", fontsize=11)ax2.set_title("KV cache memory: O(n)", fontsize=12, fontweight="bold")ax2.grid(True, alpha=0.3)fig.suptitle("Why context windows are expensive",             fontsize=14, fontweight="bold", y=1.02)plt.tight_layout()plt.show()print("At 128K tokens, attention compute is 65,536× that of 512 tokens.")print("This is why FlashAttention and ring attention exist — to tame the quadratic cost.")

### Lost in the middleA well-documented phenomenon: LLMs are best at recalling information placed at the **beginning** or**end** of the context, and worst at recalling information in the **middle**.This happens because:- Early tokens get strong attention from all later tokens (they are always "visible").- The most recent tokens get strong attention due to recency bias in the position embeddings.- Middle tokens compete with many others for attention and get diluted.This has direct implications for prompt engineering: **put critical information at the start or end**.

In [ ]:
# ── Attention weight distribution across positions (simulated) ───────────# We simulate the "attention sink" and "recency" pattern observed in real models# by averaging attention from the last token across all heads and layers.# Use the real model to demonstratelong_prompt = (    "Below is a list of facts. Read all of them carefully.\n"    + "\n".join([f"Fact {i}: The value of item {i} is {np.random.randint(100, 999)}."                 for i in range(40)])    + "\nNow, what was the value of item 20?")inputs = tokenizer(long_prompt, return_tensors="pt").to(model.device)seq_len_long = inputs["input_ids"].shape[1]print(f"Prompt length: {seq_len_long} tokens")with torch.no_grad():    outputs = model(**inputs, output_attentions=True)# Average attention FROM the last token TO all positions, across all heads/layerslast_token_attn = []for layer_attn in outputs.attentions:    # (1, n_heads, seq, seq) → take last query position, average over heads    attn_from_last = layer_attn[0, :, -1, :].mean(dim=0).cpu().float().numpy()    last_token_attn.append(attn_from_last)avg_attn = np.mean(last_token_attn, axis=0)fig, ax = plt.subplots(figsize=(14, 4))ax.bar(range(seq_len_long), avg_attn, width=1.0, color="#2196F3", alpha=0.7)ax.set_xlabel("Token position", fontsize=11)ax.set_ylabel("Avg attention weight\n(from last token)", fontsize=11)ax.set_title("Where the last token looks: attention 'sinks' at start, recency at end",             fontsize=12, fontweight="bold")ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("The U-shaped pattern: strong attention at the beginning and end, weaker in the middle.")print("This is the 'lost in the middle' phenomenon — critical for RAG context design.")

## 7 — Positional encodings: how transformers know word orderAttention is **permutation-invariant** — without any positional signal, swapping two tokens producesthe same output (just reordered). The sentence "cat the sat mat on the" would get the same attentionpattern as "The cat sat on the mat".Positional encodings inject position information so the model can distinguish order.### Two approaches| Approach | How it works | Used by ||---|---|---|| **Sinusoidal** (original transformer) | Fixed sine/cosine functions at different frequencies added to embeddings | Original Transformer, some encoder models || **RoPE** (Rotary Position Embeddings) | Rotates Q and K vectors by position-dependent angles | Llama, Qwen, Mistral, most modern LLMs |RoPE is the modern standard because it:- Encodes *relative* position (distance between tokens matters more than absolute position)- Naturally decays attention with distance (far-apart tokens attend less)- Enables context extension via interpolation (YaRN, NTK-aware scaling)

In [ ]:
# ── Visualize sinusoidal positional encodings ────────────────────────────def sinusoidal_position_encoding(max_len: int, d_model: int) -> np.ndarray:    """Generate sinusoidal positional encoding matrix."""    pe = np.zeros((max_len, d_model))    position = np.arange(max_len)[:, np.newaxis]       # (max_len, 1)    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))    pe[:, 0::2] = np.sin(position * div_term)  # even dimensions    pe[:, 1::2] = np.cos(position * div_term)  # odd dimensions    return pepe = sinusoidal_position_encoding(max_len=128, d_model=64)fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))# Heatmapim = ax1.imshow(pe.T, cmap="RdBu", aspect="auto", interpolation="nearest")ax1.set_xlabel("Position (token index)", fontsize=11)ax1.set_ylabel("Embedding dimension", fontsize=11)ax1.set_title("Sinusoidal Positional Encoding — each position has a unique pattern",              fontsize=12, fontweight="bold")fig.colorbar(im, ax=ax1, shrink=0.8)# Individual dimensionsfor dim in [0, 1, 4, 8, 16, 32]:    ax2.plot(pe[:, dim], label=f"dim {dim}", alpha=0.8)ax2.set_xlabel("Position", fontsize=11)ax2.set_ylabel("Encoding value", fontsize=11)ax2.set_title("Individual dimensions: low dims = high frequency, high dims = low frequency",              fontsize=12, fontweight="bold")ax2.legend(fontsize=9, ncol=3)ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Each position gets a unique 'fingerprint' of sine/cosine values.")print("Low dimensions oscillate fast (capture fine position), high dimensions oscillate slow (coarse).")

### RoPE: Rotary Position EmbeddingsRoPE doesn't *add* a position vector to the embedding. Instead, it **rotates** the query and key vectorsby an angle proportional to the token's position. When you compute the dot product $q_i \cdot k_j$, therotation encodes the *relative* distance $i - j$.The key property: $\langle \text{RoPE}(q, i), \text{RoPE}(k, j) \rangle$ depends only on $q$, $k$,and $i - j$ — the relative distance. This is why RoPE-based models can be extended to longer contextsby adjusting the rotation frequency (YaRN, NTK-aware interpolation).RoPE operates on pairs of dimensions, rotating each pair by:$$\theta_m = \frac{\text{position}}{10000^{2m/d}}$$This creates a spectrum of rotation speeds — fast rotation for fine position discrimination, slowrotation for coarse position.

In [ ]:
# ── Visualize RoPE rotation ─────────────────────────────────────────────def rope_freqs(dim: int, max_len: int, base: float = 10000.0) -> np.ndarray:    """Compute RoPE rotation angles for each position and dimension pair."""    freqs = 1.0 / (base ** (np.arange(0, dim, 2) / dim))    positions = np.arange(max_len)    # Outer product: (max_len, dim//2)    angles = np.outer(positions, freqs)    return anglesangles = rope_freqs(dim=64, max_len=256)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))# Rotation angles heatmapim = ax1.imshow(angles.T, cmap="twilight", aspect="auto")ax1.set_xlabel("Position", fontsize=11)ax1.set_ylabel("Dimension pair index", fontsize=11)ax1.set_title("RoPE rotation angles by position",              fontsize=12, fontweight="bold")fig.colorbar(im, ax=ax1, shrink=0.8)# Cosine similarity decay with distancedim = 64q = np.random.randn(dim)distances = np.arange(0, 200)similarities = []for dist in distances:    # Simulate how RoPE causes attention to decay with distance    angle_diff = rope_freqs(dim, dist + 1)[-1]  # angles for the distance    # Average cosine of rotation across all dimension pairs    sim = np.mean(np.cos(angle_diff))    similarities.append(sim)ax2.plot(distances, similarities, color="#E91E63", linewidth=2)ax2.set_xlabel("Token distance", fontsize=11)ax2.set_ylabel("Average cos(rotation angle)", fontsize=11)ax2.set_title("RoPE naturally decays attention with distance",              fontsize=12, fontweight="bold")ax2.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Left: low dimension pairs rotate fast, high pairs rotate slowly.")print("Right: cosine similarity decays with distance — nearby tokens are naturally more similar.")print("This is why LLMs have a natural 'locality bias' — nearby context matters more.")

## 8 — Why this matters for everything elseAttention and context are not just theory — they directly determine how you design prompts, buildretrieval systems, and serve models in production.### Prompt engineering| Principle | Why (from this notebook) ||---|---|| Put critical info at the **start** or **end** of the prompt | Lost-in-the-middle: attention is weakest for middle positions || Token order matters | Causal masking: each token can only see what came before || Shorter prompts → faster responses | Prefill time scales with $O(n^2)$ in attention || Repetition helps | Repeated tokens accumulate attention weight from multiple positions |### RAG (Retrieval-Augmented Generation)| Principle | Why ||---|---|| Context budget is limited | KV cache memory is finite; more chunks = more memory = more cost || Put the most relevant chunk first or last | Lost-in-the-middle applies to retrieved context too || Chunk size matters | Longer chunks consume more of the context window for the same information |### Agents| Principle | Why ||---|---|| Long reasoning traces hit context limits | Each thought/action/observation adds tokens || Summarize intermediate steps | Reduces context usage without losing key information || Context window size determines max trajectory length | Hard architectural limit |### Systems (serving & inference)| Principle | Why ||---|---|| KV cache is the #1 memory consumer | We showed: 8B model at 128K context → >4 GB just for KV cache || Paged attention manages KV cache like virtual memory | Avoids fragmentation, enables larger batches || Prefix caching reuses KV cache across requests | System prompts are the same across requests → cache once || Continuous batching interleaves requests at the token level | Different sequences have different lengths → dynamic scheduling |### Finetuning| Principle | Why ||---|---|| Attention patterns are learned during training | Finetuning adjusts which tokens attend to which || LoRA often targets Q and V projections | These are the attention weight matrices we built from scratch |

## Exercises### 1. Attention weight analysisFeed 5 different prompts to the Qwen3 model (e.g., a question, a code snippet, a list, a story,a chain-of-thought). For each:- Extract attention weights from 3 different layers (early, middle, late).- Pick 2 heads per layer and visualize them.- Write down: what pattern does each head seem to capture? (positional, syntactic, semantic?)### 2. KV cache calculatorBuild a function `kv_cache_report(model_config, seq_lengths, batch_sizes, dtypes)` that:- Takes a model config dict (layers, heads, head_dim, kv_heads).- Computes KV cache memory for each combination of sequence length, batch size, and dtype.- Returns a DataFrame and plots a 2D heatmap (seq_length × batch_size) for each dtype.- Test it on at least 3 model configs (e.g., 1B, 8B, 70B).### 3. Lost in the middleDesign an experiment:- Create a prompt with a list of 30 key-value facts.- Place one "target" fact at positions 1, 5, 10, 15, 20, 25, and 30.- Ask the model to retrieve the target fact.- Run each position 3 times (to reduce noise) and record accuracy.- Plot accuracy vs. position. Does the U-shaped curve from the literature appear?

In [ ]:
# ── Exercise starter code ────────────────────────────────────────────────# Exercise 1 starter: extract attention for a custom promptdef get_attention_for_prompt(prompt: str, layer: int, head: int):    """Extract attention weights for a given prompt, layer, and head."""    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)    tok_labels = [tokenizer.decode(t) for t in inputs["input_ids"][0]]    with torch.no_grad():        out = model(**inputs, output_attentions=True)    attn = out.attentions[layer][0, head].cpu().float().numpy()    return attn, tok_labels# Example usage:attn_ex, labels_ex = get_attention_for_prompt(    "The quick brown fox jumps over the lazy dog",    layer=0, head=0)print(f"Attention shape: {attn_ex.shape}")print(f"Tokens: {labels_ex}")print("\n→ Use this function to explore different prompts and heads!")# Exercise 2 starter: extend the kv_cache_memory_gb function# Hint: add batch_size parameter, loop over combinations, use pandas# Exercise 3 starter: template for lost-in-the-middle experimentdef make_needle_prompt(n_facts: int, needle_position: int, needle_key: str = "alpha") -> str:    """Create a prompt with a 'needle' fact at a specific position."""    facts = []    for i in range(n_facts):        if i == needle_position:            facts.append(f"Fact {i+1}: The secret code for {needle_key} is 42.")        else:            val = np.random.randint(100, 999)            facts.append(f"Fact {i+1}: The value of item-{chr(65+i)} is {val}.")    prompt = "Read the following facts carefully:\n" + "\n".join(facts)    prompt += f"\n\nQuestion: What is the secret code for {needle_key}?\nAnswer:"    return prompt# Exampleprint(make_needle_prompt(5, needle_position=2))

## Key Takeaways| # | Takeaway ||---|----------|| 1 | Attention lets every token directly access every other token — no information bottleneck || 2 | Self-attention projects inputs to Q, K, V and computes softmax(QKᵀ/√d_k) · V || 3 | Multi-head attention runs parallel heads, each capturing different relationships || 4 | Causal masking enforces left-to-right generation — each token sees only its past || 5 | The KV cache avoids redundant computation but consumes O(n) memory per layer || 6 | Context windows are bounded by O(n²) attention compute and O(n) KV cache memory || 7 | Positional encodings (especially RoPE) let transformers understand token order || 8 | "Lost in the middle" means critical info should go at the start or end of context || 9 | Every downstream task — prompting, RAG, agents, serving — is shaped by these mechanics |

## What's Next- **Next**: [06_scaling_laws.ipynb](06_scaling_laws.ipynb) — How model performance scales with compute, data, and parameters- **Systems track**: [systems/01_intro_to_llm_systems_2026.ipynb](../systems/01_intro_to_llm_systems_2026.ipynb) — See KV cache and paged attention in production serving- **RAG track**: [rag/simple_rag.ipynb](../rag/simple_rag.ipynb) — Apply context budget thinking to retrieval-augmented generation- **Prompt engineering**: [prompt-engineering/prompt-length-complexity-management.ipynb](../prompt-engineering/prompt-length-complexity-management.ipynb) — Practical prompt design informed by attention mechanics

## References & Further Reading1. [Vaswani et al., "Attention Is All You Need," NeurIPS 2017](https://arxiv.org/abs/1706.03762) — The paper that introduced the transformer architecture and scaled dot-product attention2. [Su et al., "RoFormer: Enhanced Transformer with Rotary Position Embedding," 2021](https://arxiv.org/abs/2104.09864) — Rotary position embeddings, now the standard in most open LLMs3. [Liu et al., "Lost in the Middle: How Language Models Use Long Contexts," 2023](https://arxiv.org/abs/2307.03172) — Empirical study of the U-shaped retrieval accuracy across context positions4. [Kwon et al., "Efficient Memory Management for Large Language Model Serving with PagedAttention," SOSP 2023](https://arxiv.org/abs/2309.06180) — PagedAttention and vLLM: treating KV cache like virtual memory5. [Dao et al., "FlashAttention: Fast and Memory-Efficient Exact Attention," NeurIPS 2022](https://arxiv.org/abs/2205.14135) — IO-aware attention that reduces the memory footprint of the O(n²) computation